# Imports

In [40]:
import os
from datetime import datetime
from dotenv import load_dotenv
from supabase import create_client, Client
from werkzeug.security import generate_password_hash, check_password_hash

# Supabase Client

In [41]:
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase: Client = create_client(
 SUPABASE_URL,
 SUPABASE_KEY
)

# TABLE: users

## Insert user

--------------------
|      users       |
--------------------
| id PK            |
| username         |
| email            |
| password_hash    |
| created_at       |



In [42]:
username = input("Username: ")
email = input("Email: ")
password = input("Password: ")
response = supabase.table("users").insert({
            'username': username,
            'email' : email,
            'password_hash' : generate_password_hash(password),
            'created_at' : datetime.today().isoformat(),
            }).execute()

## Validating User Hashed Password

In [5]:
# Checking username on database
response = supabase.table("users").select("*").eq('username', 'user_test_1_hashed').execute().data
response

[{'id': 4,
  'username': 'user_test_1_hashed',
  'email': 'user_email_1_hashed',
  'password_hash': 'scrypt:32768:8:1$DbKel2G7qXqAvw5G$162027289b2a95e40f8f9cc398050087d67ce24f1139536ce3ee004e09b130a2a40650194a94e89d311c090024bab8cce852aa9de90c1684a6f48441cce5db29',
  'created_at': '2026-06-26'}]

In [46]:
username_input = input("Username: ")
try:
    response = supabase.table("users").select("*").eq('username', username_input).execute().data
    if len(response) >0:
        password_input = input("Password: ")
        if check_password_hash(response[0]['password_hash'], password_input):
            print('Access Granted')
        else:
            print('Access Denied')
    else:
        print('USER NOT FOUND')
except Exception as e:
    error = e.__dict__
    print(error.keys())
    for i in error.keys():
        print(f'KEY: {i}\n')

Access Denied


# TABLE: entries

------------------
|     entries      |
------------------
| id_entry PK      |
| id_user FK       |
| type             |
| title            |
| content          |
| audio_url        |
| favorite         |
| created_at       |
| updated_at       |
------------------


## Insert Entry

In [52]:
id_user = 2
entry_type = "Diary" # Types: Diary, Documents, Quotes, Audio
title = "Today was a good day"
content = "BLA BLA BLA"


supabase.table("entries").insert(
    {
        "id_user" : id_user,
        "type" : entry_type,
        "title" : title,
        "content" : content,
        "audio_url" : "audio/diretory",
        "favorite" : False,
        "created_at" : datetime.today().isoformat(),
        "updated_at" : None
    }
).execute()

APIResponse(data=[{'id_entry': 4, 'id_user': 2, 'type': 'Diary', 'title': 'Today was a good day', 'content': 'BLA BLA BLA', 'audio_url': 'audio/diretory', 'favorite': False, 'created_at': '2026-06-27', 'updated_at': None}], count=None)

## Update entry

In [55]:
id_entry = 2
title = "This entry was updated via backend"
content = "First test updating"

supabase.table("entries").update(
        {
        "title" : title,
        "content" : content,
        "audio_url" : "audio/diretory",
        "favorite" : False,
        "updated_at" : datetime.today().isoformat()
    }
).eq("id_entry", id_entry).execute()

APIResponse(data=[{'id_entry': 2, 'id_user': 4, 'type': 'Diary', 'title': 'This entry was updated via backend', 'content': 'First test updating', 'audio_url': 'audio/diretory', 'favorite': False, 'created_at': '2026-06-27', 'updated_at': '2026-06-27'}], count=None)

# TABLE: tags

------------------
|    entry_tags    |
------------------
| entry_id FK      |
| tag_id FK        |
------------------

------------------
|       tags       |
------------------
| id PK            |
| name             |
------------------

## Inserting tag

In [79]:
# tag = "Family"
def insert_tag(tag):
    try:
        response = supabase.table("tags").insert({
            "tag" : tag
        }).execute()
        print(response)
        return response.data
    except Exception as e:
        print(e.__dict__['message'])

In [80]:
response = insert_tag("TAG TEST 1")

data=[{'id_tag': 11, 'tag': 'TAG TEST 1'}] count=None


In [82]:
response[0]['id_tag']

11

## Entry x Tag Relation

In [110]:
# id_entry = 2
# id_tag = 1
def create_entry_tag_relation(id_entry, id_tag):
    try:
        response = supabase.table("entry_tags").insert({
            "id_entry" : id_entry,
            "id_tag" : id_tag
        }).execute()
        # print(response)
        return response
    except Exception as e:
        print(e.__dict__['message'])

## Checking if Entry x Tag Relation exists

In [106]:
id_entry = 2
id_tag = 1
def check_entry_tag_relation_existance(id_entry, id_tag):

    response = supabase.table("entry_tags").select("*").eq('id_entry', id_entry).eq('id_tag', id_tag).execute().data
    return len(response) == 1

In [112]:
check_entry_tag_relation_existance(2,13)

False

## Tagging logic

In [121]:
# First, search if tag exists. If it doesn't exist, it creates it before relating it to an entry.
tag_name = "Giovanna"
id_entry = 3
try:
    tag = supabase.table("tags").select("*").eq("tag", tag_name).execute().data
    if len(tag) == 0:
        tag = insert_tag(tag_name)

    if check_entry_tag_relation_existance(id_entry, tag[0]['id_tag']):
        print(f"This entry is already tagged with {tag_name}")
    else:
        response = create_entry_tag_relation(id_entry, tag[0]['id_tag'])
        print(response)

except Exception as e:
    print(f"ERROR: {e.__dict__['message']}")

This entry is already tagged with Giovanna


In [77]:
tag = "Family"
tag = supabase.table("tags").select("*").eq("tag", tag).execute().data
print(tag[0]['id_tag'])

10
